In [ ]:
%%sql -r dataframe_1
CREATE WAREHOUSE IF NOT EXISTS CUSTOMER_WH_11
    WITH WAREHOUSE_SIZE = 'XSMALL'
         AUTO_SUSPEND   = 60
         AUTO_RESUME    = TRUE
         INITIALLY_SUSPENDED = TRUE;

In [ ]:
%%sql -r dataframe_2
USE WAREHOUSE CUSTOMER_WH_11;

In [ ]:
%%sql -r dataframe_3
CREATE DATABASE IF NOT EXISTS CUSTOMER_MDM_DB_11;

In [ ]:
%%sql -r dataframe_4
CREATE SCHEMA IF NOT EXISTS CUSTOMER_MDM_DB_11.CUSTOMER_SCHEMA_11;


In [ ]:
%%sql -r dataframe_5
USE DATABASE CUSTOMER_MDM_DB_11;


In [ ]:
%%sql -r dataframe_6
USE SCHEMA CUSTOMER_SCHEMA_11;


In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE TABLE CUSTOMER_HYBRID_DIM (
    CUSTOMER_KEY            NUMBER AUTOINCREMENT START 1 INCREMENT 1 PRIMARY KEY,
    CUSTOMER_ID             NUMBER         NOT NULL,
    CUSTOMER_NAME           VARCHAR(100)   NOT NULL,
    CITY                    VARCHAR(100),
    PREVIOUS_CITY           VARCHAR(100),
    STATE                   VARCHAR(100),
    CURRENT_MEMBERSHIP      VARCHAR(50),
    PREVIOUS_MEMBERSHIP     VARCHAR(50),
    HISTORICAL_MEMBERSHIP   VARCHAR(50),
    SEGMENT                 VARCHAR(50),
    EFFECTIVE_DATE          DATE           NOT NULL,
    EXPIRY_DATE             DATE           NOT NULL,
    IS_CURRENT              BOOLEAN        NOT NULL DEFAULT TRUE
);

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE FILE FORMAT CSV_FORMAT_11
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    NULL_IF = ('NULL', '');

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE STAGE CUSTOMER_STAGE_11
    FILE_FORMAT = CSV_FORMAT_11;

In [ ]:
%%sql -r dataframe_10
CREATE OR REPLACE TABLE CUSTOMER_INITIAL_STG (
    CUSTOMER_ID    NUMBER,
    CUSTOMER_NAME  VARCHAR(100),
    CITY           VARCHAR(100),
    STATE          VARCHAR(100),
    MEMBERSHIP     VARCHAR(50),
    SEGMENT        VARCHAR(50)
);


In [ ]:
%%sql -r dataframe_11
COPY INTO CUSTOMER_INITIAL_STG
    FROM @CUSTOMER_STAGE_11/customers_initial.csv
    FILE_FORMAT = (FORMAT_NAME = CSV_FORMAT_11)
    ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
%%sql -r dataframe_12
INSERT INTO CUSTOMER_HYBRID_DIM
    (CUSTOMER_ID, CUSTOMER_NAME, CITY, PREVIOUS_CITY, STATE,
     CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP, HISTORICAL_MEMBERSHIP,
     SEGMENT, EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT)
SELECT
    CUSTOMER_ID, CUSTOMER_NAME, CITY, NULL, STATE,
    MEMBERSHIP, NULL, MEMBERSHIP,
    SEGMENT, '2026-01-01'::DATE, '9999-12-31'::DATE, TRUE
FROM CUSTOMER_INITIAL_STG;

SELECT 'Initial Hybrid Dimension Data Loaded Successfully' AS STATUS,
       COUNT(*)             AS TOTAL_RECORDS,
       COUNT_IF(IS_CURRENT) AS CURRENT_RECORDS
FROM CUSTOMER_HYBRID_DIM;

In [ ]:
%%sql -r dataframe_13
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, PREVIOUS_CITY, STATE,
       CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP, HISTORICAL_MEMBERSHIP,
       SEGMENT, EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT
FROM CUSTOMER_HYBRID_DIM
ORDER BY CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_14
CREATE OR REPLACE TABLE CUSTOMER_UPDATES_STG (
    CUSTOMER_ID     NUMBER,
    CUSTOMER_NAME   VARCHAR(100),
    CITY            VARCHAR(100),
    STATE           VARCHAR(100),
    MEMBERSHIP      VARCHAR(50),
    SEGMENT         VARCHAR(50),
    EFFECTIVE_DATE  DATE
);

In [ ]:
%%sql -r dataframe_15
LIST @CUSTOMER_STAGE_11;

In [ ]:
%%sql -r dataframe_16
COPY INTO CUSTOMER_UPDATES_STG
    FROM @CUSTOMER_STAGE_11/customer_updates.csv
    FILE_FORMAT = (FORMAT_NAME = CSV_FORMAT_11)
    ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
%%sql -r dataframe_17
CREATE OR REPLACE TEMPORARY TABLE PRE_UPDATE_SNAPSHOT AS
SELECT d.CUSTOMER_ID, d.CITY AS OLD_CITY, d.CURRENT_MEMBERSHIP AS OLD_MEMBERSHIP
FROM CUSTOMER_HYBRID_DIM d
JOIN CUSTOMER_UPDATES_STG u ON u.CUSTOMER_ID = d.CUSTOMER_ID
WHERE d.IS_CURRENT = TRUE;

In [ ]:
%%sql -r dataframe_18
UPDATE CUSTOMER_HYBRID_DIM t
SET EXPIRY_DATE = DATEADD(day, -1, u.EFFECTIVE_DATE),
    IS_CURRENT  = FALSE
FROM CUSTOMER_UPDATES_STG u
WHERE t.CUSTOMER_ID = u.CUSTOMER_ID
  AND t.IS_CURRENT = TRUE;

In [ ]:
%%sql -r dataframe_19
INSERT INTO CUSTOMER_HYBRID_DIM
    (CUSTOMER_ID, CUSTOMER_NAME, CITY, PREVIOUS_CITY, STATE,
     CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP, HISTORICAL_MEMBERSHIP,
     SEGMENT, EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT)
SELECT
    u.CUSTOMER_ID, u.CUSTOMER_NAME, u.CITY, p.OLD_CITY, u.STATE,
    u.MEMBERSHIP, p.OLD_MEMBERSHIP, u.MEMBERSHIP,
    u.SEGMENT, u.EFFECTIVE_DATE, '9999-12-31'::DATE, TRUE
FROM CUSTOMER_UPDATES_STG u
JOIN PRE_UPDATE_SNAPSHOT p ON p.CUSTOMER_ID = u.CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_20
UPDATE CUSTOMER_HYBRID_DIM t
SET t.CITY                = n.CITY,
    t.PREVIOUS_CITY       = n.PREVIOUS_CITY,
    t.STATE                = n.STATE,
    t.CURRENT_MEMBERSHIP  = n.CURRENT_MEMBERSHIP,
    t.PREVIOUS_MEMBERSHIP = n.PREVIOUS_MEMBERSHIP
FROM (SELECT CUSTOMER_ID, CITY, PREVIOUS_CITY, STATE, CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP
      FROM CUSTOMER_HYBRID_DIM
      WHERE IS_CURRENT = TRUE) n
WHERE t.CUSTOMER_ID = n.CUSTOMER_ID
  AND t.CUSTOMER_ID IN (SELECT CUSTOMER_ID FROM CUSTOMER_UPDATES_STG);


In [ ]:
%%sql -r dataframe_21
SELECT 'Hybrid SCD Updates Applied Successfully' AS STATUS,
       COUNT(*)                 AS TOTAL_RECORDS,
       COUNT_IF(IS_CURRENT)     AS CURRENT_RECORDS,
       COUNT_IF(NOT IS_CURRENT) AS HISTORICAL_RECORDS
FROM CUSTOMER_HYBRID_DIM;

In [ ]:
%%sql -r dataframe_22
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, PREVIOUS_CITY, STATE,
       CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP, HISTORICAL_MEMBERSHIP,
       SEGMENT, EFFECTIVE_DATE, EXPIRY_DATE, IS_CURRENT
FROM CUSTOMER_HYBRID_DIM
ORDER BY CUSTOMER_ID, EFFECTIVE_DATE;


In [ ]:
%%sql -r dataframe_23
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, PREVIOUS_CITY, STATE,
       CURRENT_MEMBERSHIP, PREVIOUS_MEMBERSHIP, SEGMENT
FROM CUSTOMER_HYBRID_DIM
WHERE IS_CURRENT = TRUE
ORDER BY CUSTOMER_ID;

In [ ]:
%%sql -r dataframe_24
SELECT CUSTOMER_ID, CUSTOMER_NAME, CITY, HISTORICAL_MEMBERSHIP, SEGMENT,
       EFFECTIVE_DATE, EXPIRY_DATE
FROM CUSTOMER_HYBRID_DIM
WHERE CUSTOMER_ID = 101
  AND '2026-03-15'::DATE BETWEEN EFFECTIVE_DATE AND EXPIRY_DATE;


In [ ]:
%%sql -r dataframe_25
SELECT 'TOTAL RECORD COUNT' AS METRIC, COUNT(*) AS VALUE FROM CUSTOMER_HYBRID_DIM
UNION ALL
SELECT 'CURRENT RECORD COUNT', COUNT_IF(IS_CURRENT) FROM CUSTOMER_HYBRID_DIM
UNION ALL
SELECT 'HISTORICAL RECORD COUNT', COUNT_IF(NOT IS_CURRENT) FROM CUSTOMER_HYBRID_DIM;